# BSL Gesture Recognition – CPU Latency Benchmark (Colab)

This notebook evaluates **end-to-end inference latency and throughput** of different models on CPU.  
The goal is to provide reproducible and publication-ready benchmarking results.

---

## Models Under Test
- **Deep Learning:** 1D-CNN, Transformer-Encoder, ADANN  
- **Machine Learning:** LightGBM, XGBoost  
- **Hybrid:** ADANN + LightGBM (DA–LGBM)

---

## Methodology
1. Standardised input shape and data type across models  
2. Warm-up runs (to stabilise cache/JIT compilation)  
3. Perform *N* measurement runs, record **mean latency** and **throughput**  
4. Export plots in publication-ready format (PNG 300 dpi, PDF)  
5. Save raw measurement results to CSV for reproducibility  

# ===== Section 1: Minimal deps (CPU-only) =====

In [ ]:
!pip -q install --upgrade pip
!pip -q install numpy pandas

import sys, numpy, pandas
print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.2 MB/s eta 0:00:00
Python: 3.12.11
NumPy: 2.0.2
Pandas: 2.2.2


# ===== Section 2: CPU env (lock threads, report) =====

In [ ]:
import os, platform, json, re, time, psutil, numpy as np, random
os.environ["CUDA_VISIBLE_DEVICES"] = ""     # no GPU
os.environ["OMP_NUM_THREADS"] = "1"         # deterministic CPU
SEED = 1234
np.random.seed(SEED); random.seed(SEED)

def _cpu_info():
    info = {"platform": platform.platform(), "machine": platform.machine()}
    try:
        with open("/proc/cpuinfo","r") as f: txt=f.read()
        m = re.search(r"model name\s*:\s*(.+)", txt); info["cpu_model"] = m.group(1).strip() if m else platform.processor()
    except: info["cpu_model"] = platform.processor()
    info.update({"logical_cpus": psutil.cpu_count(True),
                 "physical_cpus": psutil.cpu_count(False),
                 "omp_num_threads": os.getenv("OMP_NUM_THREADS","1"),
                 "time": time.strftime("%Y-%m-%d %H:%M:%S")})
    return info

print(json.dumps(_cpu_info(), indent=2))


{
  "platform": "Linux-6.1.123+-x86_64-with-glibc2.35",
  "machine": "x86_64",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.20GHz",
  "logical_cpus": 2,
  "physical_cpus": 1,
  "omp_num_threads": "1",
  "time": "2025-08-23 20:21:18"
}


# ===== Section 3: Paths + pick one CSV =====

In [ ]:
# ===== Section 3: Paths + robust CSV loader (skip comments / pad to 100x5) =====
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, numpy as np, pandas as pd
PROJECT_ROOT = "/content/drive/My Drive/Project"   # <- change if needed
os.chdir(PROJECT_ROOT)
print("Working dir:", os.getcwd())

MODELS_DIR = "models"                    # matches your screenshot
CSV_DIR    = "datasets/gesture_csv"      # put your CSVs here (any mixed delimiters OK)

SEED = 1234
rng  = np.random.default_rng(SEED)

def _sniff_read(path):
    """Try several separators; if all fail, do a manual regex split."""
    tried = []
    for kwargs in [
        dict(engine='python', sep=None,           comment='#', skip_blank_lines=True),
        dict(engine='python', sep=r'[,\t; ]+',    comment='#', skip_blank_lines=True),
        dict(engine='python', sep=',',            comment='#', skip_blank_lines=True),
        dict(engine='python', sep=';',            comment='#', skip_blank_lines=True),
        dict(engine='python', sep='\t',           comment='#', skip_blank_lines=True),
        dict(engine='python', delim_whitespace=True, comment='#', skip_blank_lines=True),
    ]:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[1] >= 5: return df
            tried.append(str(kwargs))
        except Exception as e:
            tried.append(f"{kwargs} -> {e}")
    # manual fallback
    rows = []
    with open(path, 'r', errors='ignore') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'): continue
            s = s.replace('\t',' ').replace('|',' ')
            s = re.sub(r'\s+', ' ', s)
            parts = re.split(r'[,\s;]+', s)
            rows.append(parts)
            if len(rows) >= 300: break
    if not rows:
        raise ValueError("No data lines after stripping comments.")
    maxc = max(len(r) for r in rows)
    data = [r + ['']*(maxc-len(r)) for r in rows]
    df = pd.DataFrame(data)
    for c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
    if df.shape[1] < 5:
        raise ValueError(f"Still <5 columns. Tried: {tried}")
    return df

def _numeric_cols(df: pd.DataFrame):
    """Columns mostly numeric; drop timestamp-like first col."""
    frac = {}
    for c in df.columns:
        col = pd.to_numeric(df[c], errors='coerce')
        frac[c] = float(col.notna().mean())
    num_cols = [c for c in df.columns if frac[c] >= 0.8]
    first = str(df.columns[0]).lower()
    if ('time' in first or 'stamp' in first) and df.columns[0] in num_cols:
        num_cols = num_cols[1:]
    return num_cols

PAD_MODE = "pad_last"  # "pad_last" | "loop" | "zero"

def _fix_len_100(arr, mode=PAD_MODE):
    n = arr.shape[0]
    if n == 100: return arr.astype(np.float32, copy=False)
    if n > 100:  return arr[:100].astype(np.float32, copy=False)
    # n < 100
    if mode == "pad_last":
        last = arr[-1:, :]
        pad  = np.repeat(last, 100 - n, axis=0)
        out  = np.vstack([arr, pad])
    elif mode == "loop":
        reps = int(np.ceil(100 / n))
        out  = np.tile(arr, (reps, 1))[:100, :]
    else:  # zero
        pad  = np.zeros((100 - n, arr.shape[1]), dtype=arr.dtype)
        out  = np.vstack([arr, pad])
    return out.astype(np.float32, copy=False)

def load_window_csv_flex(path, window_len=100):
    """
    - skip '#' lines / blanks
    - robust sep detection
    - prefer named columns: thumb,index,middle,ring,pinky (case-insensitive)
    - else take last 5 numeric columns
    - ffill/bfill NaNs, then pad/trim to 100 rows
    """
    df = _sniff_read(path)
    names = [str(c).strip() for c in df.columns]
    lower = [c.lower() for c in names]
    want  = ['thumb','index','middle','ring','pinky']

    if all(w in lower for w in want):
        cols = [names[lower.index(w)] for w in want]
        df   = df[cols].copy()
    else:
        num_cols = _numeric_cols(df)
        if len(num_cols) < 5:
            df = df.apply(pd.to_numeric, errors='coerce')
            num_cols = _numeric_cols(df)
        assert len(num_cols) >= 5, f"Not enough numeric columns in {os.path.basename(path)}"
        cols = num_cols[-5:]
        df   = df[cols].copy()

    df = df.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    arr = df.to_numpy(dtype=np.float32)
    arr = _fix_len_100(arr, mode=PAD_MODE)
    assert arr.shape == (100,5), f"Expect (100,5), got {arr.shape}"
    return arr, cols

# pick ONE csv (reproducible); if a file fails, try next
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")))
assert csv_files, f"No CSV found in {CSV_DIR}"
pick_idx = int(rng.integers(0, len(csv_files)))
RAW_WIN = None
for k in range(len(csv_files)):
    CSV_PICK = csv_files[(pick_idx + k) % len(csv_files)]
    try:
        RAW_WIN, USED_COLS = load_window_csv_flex(CSV_PICK)
        print("Using CSV:", CSV_PICK)
        print("Using columns:", USED_COLS)
        break
    except Exception as e:
        print(f"[Skip] {os.path.basename(CSV_PICK)} -> {e}")
        continue
assert RAW_WIN is not None, "Failed to parse any CSV in the folder."


Mounted at /content/drive
Working dir: /content/drive/My Drive/Project
Using CSV: datasets/gesture_csv/user_006_gesture_8_Eight_sample_6_20250724_211637.csv
Using columns: ['thumb', 'index', 'middle', 'ring', 'pinky']


# ===== Section 4: C header runner + fusion + timing =====

In [ ]:
# ===== Section 4: TFLite runner + C-header runner + timing =====
import subprocess, textwrap, ctypes, tempfile, shutil
from time import perf_counter
import numpy as np
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # CPU-only

# ---------- TFLite (CNN / Transformer) ----------
class TFLiteRunner:
    def __init__(self, tflite_path):
        self.interp = tf.lite.Interpreter(model_path=tflite_path, num_threads=1)
        self.interp.allocate_tensors()
        self.inp = self.interp.get_input_details()[0]
        self.out = self.interp.get_output_details()[0]
        self.dtype = self.inp["dtype"]      # np.float32 / np.int8
        self.shape = list(self.inp["shape"])  # e.g. [1,100,5] or [1,100,5,1]

    def prepare_input(self, raw_win_100x5):
        x = raw_win_100x5.astype(np.float32)
        # no external scaler as you requested; we pass raw (models may normalize internally)
        if str(self.dtype) == "int8":
            qp = self.inp["quantization_parameters"]
            scale = float(qp["scales"][0]) if len(qp["scales"]) else 1.0
            zp    = int(qp["zero_points"][0]) if len(qp["zero_points"]) else 0
            xq = np.round(x/scale + zp).astype(np.int8)
            arr = xq
        else:
            arr = x
        # reshape
        if len(self.shape) == 3:   # [1,100,5]
            arr = arr.reshape(self.shape)
        elif len(self.shape) == 4: # [1,100,5,1] etc.
            arr = arr.reshape(self.shape)
        else:
            raise ValueError(f"Unexpected input shape: {self.shape}")
        return arr

    def call(self, arr):
        self.interp.set_tensor(self.inp["index"], arr)
        self.interp.invoke()
        _ = self.interp.get_tensor(self.out["index"])

# ---------- C headers (XGB / LGBM / ADANN / DA-LGBM branches) ----------
class CHeaderRunner:
    """
    Wrapper for .h models compiled to .so on-the-fly.

    - LightGBM / XGBoost:
        Include both model.h + bsl_model_Transformer.h
        (to reuse extract_adann_intelligent_features + normalize_adann_features).
    - ADANN:
        Use macro renaming so ADANN.h and Transformer.h can coexist.
        Then feed ADANN with the same 190-dim normalized features.
    """

    def __init__(self, header_path, transformer_header_path=None, cxx_standard="c++17"):
        import tempfile, os, subprocess, textwrap, shutil
        self._shutil = shutil
        self.tmpdir = tempfile.mkdtemp(prefix="cmodel_")
        self.header_path = os.path.abspath(header_path)
        self.tr_header  = (os.path.abspath(transformer_header_path)
                           if transformer_header_path and os.path.exists(transformer_header_path)
                           else None)
        self.so_path   = os.path.join(self.tmpdir, "libmodel.so")
        wrapper_cpp    = os.path.join(self.tmpdir, "wrapper.cpp")

        base = os.path.basename(self.header_path).lower()
        if "lightgbm" in base:
            model_kind = "lgbm"
        elif "xgboost" in base or "xgb" in base:
            model_kind = "xgb"
        elif "adann" in base:
            model_kind = "adann"
        else:
            raise RuntimeError(f"Unknown model kind from header name: {base}")

        # Build wrapper code
        if model_kind in ("lgbm", "xgb"):
            if not self.tr_header:
                # try same folder
                cand = os.path.join(os.path.dirname(self.header_path), "bsl_model_Transformer.h")
                if os.path.exists(cand): self.tr_header = cand
            if not self.tr_header:
                raise RuntimeError("Need bsl_model_Transformer.h for LGBM/XGB.")

            code = f"""
            #include <stdint.h>
            #include <math.h>
            #include <string.h>
            #include "{self.header_path.replace('\\', '\\\\')}"
            #include "{self.tr_header.replace('\\', '\\\\')}"   // feature helpers

            extern void extract_adann_intelligent_features(float* sensor_data, float* features);
            extern void normalize_adann_features(float* features, float* normalized_features);

            extern "C" int  model_num_classes(void) {{ return 11; }}

            static inline void pack500(const float in_[100][5], float out500[500]) {{
                for (int t=0;t<100;++t) for (int ch=0; ch<5; ++ch) out500[t*5+ch] = in_[t][ch];
            }}

            extern "C" void model_infer_100x5(const float in_[100][5], float out_probs[]) {{
                float raw500[500]; pack500(in_, raw500);
                float feat190[190]; extract_adann_intelligent_features(raw500, feat190);
                float f190n[190];   normalize_adann_features(feat190, f190n);
            """
            if model_kind == "lgbm":
                code += r"""
                // LightGBM: score(double*, double*)
                double in_d[190]; for(int i=0;i<190;++i) in_d[i] = (double)f190n[i];
                double out_d[11];
                score(in_d, out_d);
                double sum = 0.0; for(int i=0;i<11;++i) sum += out_d[i];
                if (sum <= 0.0) {
                    double mx = out_d[0]; for(int i=1;i<11;++i) if(out_d[i]>mx) mx=out_d[i];
                    double s=0.0; for(int i=0;i<11;++i){ out_d[i]=exp(out_d[i]-mx); s+=out_d[i]; }
                    for(int i=0;i<11;++i) out_probs[i] = (float)(out_d[i]/s);
                } else {
                    for(int i=0;i<11;++i) out_probs[i] = (float)out_d[i];
                }
                """
            else:  # xgboost
                code += r"""
                using namespace Eloquent::ML::Port;
                XGBClassifier clf;
                int cls = clf.predict(f190n);
                for (int i=0;i<11;++i) out_probs[i] = 0.0f;
                if (cls>=0 && cls<11) out_probs[cls] = 1.0f;
                """
            code += r"}"

        else:
            # --- ADANN branch with macro renaming ---
            if not self.tr_header:
                cand = os.path.join(os.path.dirname(self.header_path), "bsl_model_Transformer.h")
                if os.path.exists(cand): self.tr_header = cand
            if not self.tr_header:
                raise RuntimeError("Need bsl_model_Transformer.h for ADANN as feature helper.")

            code = f"""
            #include <stdint.h>
            #include <math.h>
            #include <string.h>

            // rename ADANN symbols to avoid collision
            #define ADANN_INPUT_SIZE   ADANN_INPUT_SIZE_ADH
            #define adann_scaler_mean  adann_scaler_mean_ADH
            #define adann_scaler_scale adann_scaler_scale_ADH
            #define adann_predict      adann_predict_ADH

            #include "{self.header_path.replace('\\', '\\\\')}"   // ADANN.h

            #undef ADANN_INPUT_SIZE
            #undef adann_scaler_mean
            #undef adann_scaler_scale
            #undef adann_predict

            #include "{self.tr_header.replace('\\', '\\\\')}"     // Transformer.h for features

            extern void extract_adann_intelligent_features(float* sensor_data, float* features);
            extern void normalize_adann_features(float* features, float* normalized_features);

            extern "C" int  model_num_classes(void) {{ return 11; }}

            static inline void pack500(const float in_[100][5], float out500[500]) {{
                for (int t=0;t<100;++t) for (int ch=0; ch<5; ++ch) out500[t*5+ch] = in_[t][ch];
            }}

            extern "C" void model_infer_100x5(const float in_[100][5], float out_probs[]) {{
                float raw500[500]; pack500(in_, raw500);
                float feat190[190]; extract_adann_intelligent_features(raw500, feat190);
                float f190n[190];   normalize_adann_features(feat190, f190n);

                int cls = adann_predict_ADH(f190n); // call renamed ADANN function
                for (int i=0;i<11;++i) out_probs[i] = 0.0f;
                if (cls>=0 && cls<11) out_probs[cls] = 1.0f;
            }}
            """

        # --- write wrapper.cpp & compile ---
        with open(wrapper_cpp, "w") as f: f.write(textwrap.dedent(code))
        cmd = ["g++", f"-std={cxx_standard}", "-O3", "-fPIC", "-shared", wrapper_cpp, "-o", self.so_path, "-lm"]
        import subprocess
        try:
            subprocess.check_output(cmd, stderr=subprocess.STDOUT)
        except subprocess.CalledProcessError as e:
            print(">>> g++ command:\n", " ".join(cmd))
            out = e.output.decode("utf-8", errors="ignore")
            print(">>> g++ error (first 200 lines):\n", "\n".join(out.splitlines()[:200]))
            raise

        # ctypes binding
        import ctypes
        self.lib = ctypes.CDLL(self.so_path)
        self.lib.model_num_classes.restype = ctypes.c_int
        Float5      = ctypes.c_float * 5
        Float100x5  = Float5 * 100
        self._Float100x5 = Float100x5
        self.lib.model_infer_100x5.argtypes = [ctypes.POINTER(Float100x5), ctypes.POINTER(ctypes.c_float)]
        self.lib.model_infer_100x5.restype  = None
        self.num_classes = self.lib.model_num_classes()

    def infer(self, raw_win_100x5):
        import numpy as np, ctypes
        assert raw_win_100x5.shape == (100,5)
        buf_in = self._Float100x5()
        for t in range(100):
            for ch in range(5):
                buf_in[t][ch] = float(raw_win_100x5[t, ch])
        out = (ctypes.c_float * self.num_classes)()
        self.lib.model_infer_100x5(buf_in, out)
        return np.array([out[i] for i in range(self.num_classes)], dtype=np.float32)

    def close(self):
        try: self._shutil.rmtree(self.tmpdir)
        except: pass


# ---------- timing ----------
def time_infer(call, warmup=20, runs=500):
    for _ in range(warmup): call()
    xs=[]
    for _ in range(runs):
        t0=perf_counter(); call(); xs.append((perf_counter()-t0)*1000.0)
    xs_sorted = sorted(xs)
    p50 = xs_sorted[int(round(0.50*(runs-1)))]
    p95 = xs_sorted[int(round(0.95*(runs-1)))]
    return {"mean_ms": float(np.mean(xs_sorted)), "p50_ms": float(p50), "p95_ms": float(p95), "runs": runs}


#===== Section 5: Model headers + run =====

In [ ]:
# ===== Section 5: Run and compare =====
import os, pandas as pd

# Files (exactly as in your screenshot)
PATH_TFLITE_CNN   = os.path.join(MODELS_DIR, "bsl_model_1D_CNN.tflite")
PATH_TFLITE_TRF   = os.path.join(MODELS_DIR, "bsl_model_Transformer.tflite")
PATH_HDR_XGB      = os.path.join(MODELS_DIR, "bsl_model_XGBoost.h")
PATH_HDR_LGBM     = os.path.join(MODELS_DIR, "bsl_model_LightGBM.h")
PATH_HDR_ADANN    = os.path.join(MODELS_DIR, "bsl_model_ADANN.h")
PATH_HDR_DA_ADANN = os.path.join(MODELS_DIR, "bsl_model_DA_LGBM_ADANN.h")
PATH_HDR_DA_LGBM  = os.path.join(MODELS_DIR, "bsl_model_DA_LGBM_LightGBM.h")
PATH_HDR_TRF_H    = os.path.join(MODELS_DIR, "bsl_model_Transformer.h")  # for feature helpers

WARMUP, RUNS = 20, 500
results = []

# --- CNN (TFLite) ---
if os.path.exists(PATH_TFLITE_CNN):
    r = TFLiteRunner(PATH_TFLITE_CNN)
    arr = r.prepare_input(RAW_WIN)
    stats = time_infer(lambda: r.call(arr), warmup=WARMUP, runs=RUNS)
    results.append({"model":"1D_CNN (TFLite)", "precision":"INT8/FP32", **stats, "path":PATH_TFLITE_CNN})
else:
    print("[Skip] CNN tflite missing:", PATH_TFLITE_CNN)

# --- Transformer (TFLite) ---
if os.path.exists(PATH_TFLITE_TRF):
    r = TFLiteRunner(PATH_TFLITE_TRF)
    arr = r.prepare_input(RAW_WIN)
    stats = time_infer(lambda: r.call(arr), warmup=WARMUP, runs=RUNS)
    results.append({"model":"Transformer (TFLite)", "precision":"INT8/FP32", **stats, "path":PATH_TFLITE_TRF})
else:
    print("[Skip] Transformer tflite missing:", PATH_TFLITE_TRF)

# --- C headers (need Transformer.h for feature helpers) ---
if not os.path.exists(PATH_HDR_TRF_H):
    print("[FATAL] Missing", PATH_HDR_TRF_H, " (needed for feature extraction).")

def run_header(name, path):
    if not (os.path.exists(path) and os.path.exists(PATH_HDR_TRF_H)):
        print(f"[Skip] {name}: missing {path} or feature header {PATH_HDR_TRF_H}")
        return None, None
    runner = CHeaderRunner(path, transformer_header_path=PATH_HDR_TRF_H)
    stats  = time_infer(lambda: runner.infer(RAW_WIN), warmup=WARMUP, runs=RUNS)
    runner.close()
    return stats, path

for name, p in [
    ("XGBoost (micromigen)", PATH_HDR_XGB),
    ("LightGBM (m2cgen)",    PATH_HDR_LGBM),
    ("ADANN (pure C)",       PATH_HDR_ADANN),
    ("DA-LGBM: ADANN-branch",PATH_HDR_DA_ADANN),
    ("DA-LGBM: LGBM-branch", PATH_HDR_DA_LGBM),
]:
    stats, path = run_header(name, p)
    if stats: results.append({"model": name, "precision":"FP32", **stats, "path": path})

# --- DA-LGBM (fused) end-to-end latency: ADANN infer + LGBM infer + fuse ---
if os.path.exists(PATH_HDR_DA_ADANN) and os.path.exists(PATH_HDR_DA_LGBM) and os.path.exists(PATH_HDR_TRF_H):
    r_ad = CHeaderRunner(PATH_HDR_DA_ADANN, transformer_header_path=PATH_HDR_TRF_H)
    r_lg = CHeaderRunner(PATH_HDR_DA_LGBM,  transformer_header_path=PATH_HDR_TRF_H)

    def _call_dalgbm_fused():
        p1 = r_ad.infer(RAW_WIN)   # ADANN probs
        p2 = r_lg.infer(RAW_WIN)   # LGBM probs
        _  = 0.5*p1 + 0.5*p2       # or your confidence fusion；不计时也几乎为0

    stats = time_infer(_call_dalgbm_fused, warmup=WARMUP, runs=RUNS)
    results.append({"model":"DA-LGBM (ADANN+LGBM fused)", "precision":"FP32", **stats, "path":"ADANN+LGBM"})
    r_ad.close(); r_lg.close()

# Save & show
df = pd.DataFrame(results)
out_csv = "latency_LOSO_inference_only_ALL.csv"
df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
if not df.empty:
    print(df[["model","mean_ms","p50_ms","p95_ms","runs","path"]].to_string(index=False))


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Saved: latency_LOSO_inference_only_ALL.csv
                     model  mean_ms   p50_ms   p95_ms  runs                                path
           1D_CNN (TFLite) 0.117895 0.107538 0.165715   500      models/bsl_model_1D_CNN.tflite
      Transformer (TFLite) 0.485332 0.465060 0.568809   500 models/bsl_model_Transformer.tflite
      XGBoost (micromigen) 0.232101 0.227445 0.253411   500          models/bsl_model_XGBoost.h
         LightGBM (m2cgen) 0.237849 0.229805 0.281591   500         models/bsl_model_LightGBM.h
            ADANN (pure C) 0.318258 0.305815 0.384188   500            models/bsl_model_ADANN.h
     DA-LGBM: ADANN-branch 0.352774 0.314142 0.539866   500    models/bsl_model_DA_LGBM_ADANN.h
      DA-LGBM: LGBM-branch 0.228030 0.210351 0.378872   500 models/bsl_model_DA_LGBM_LightGBM.h
DA-LGBM (ADANN+LGBM fused) 0.547489 0.536677 0.629176   500                          ADANN+LGBM
